In [1]:
from asyncio import run

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
import torch
from deepsnap.dataset import GraphDataset

from src.architectures import HomoGNN
from src.config import MPL_STYLE_DIR, PLOTS_DIR, PROJECT_ROOT
from src.data import DatasetLoader, GraphBuilder, Preprocessor
from src.db import PBWarehouse
from src.inference import InferenceEngine, InferenceResults
from src.models import Features, Graph, Tweet, User
from src.scraper import TweetyScraper
from src.service import UserService

plt.style.use(MPL_STYLE_DIR / "iragca_cmr10.mplstyle")

2025-11-21 14:40:10.176 | INFO     | src.config:<module>:28 - Loaded environment variables from /home/iragca/Documents/github/capstone-project-2/.env
2025-11-21 14:40:10.176 | INFO     | src.config:<module>:67 - PROJECT_ROOT: /home/iragca/Documents/github/capstone-project-2
2025-11-21 14:40:10.176 | INFO     | src.config:<module>:68 - DATA_DIR: /home/iragca/Documents/github/capstone-project-2/data


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_model():
    model_path = PROJECT_ROOT / "best_model.pth"
    # NOTE: Make sure to adjust this whenever changing model architecture
    model = HomoGNN(input_size=5, hidden_size=8, num_layers=4).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    return model


def load_data(_pb: PBWarehouse):
    dataset_loader = DatasetLoader(_pb)
    data = dataset_loader.load_dataset()
    preprocessor = Preprocessor(data=data)
    return preprocessor.preprocess()


model: HomoGNN = load_model()
pb: PBWarehouse = PBWarehouse()
scraper: TweetyScraper = TweetyScraper(previous_session=True)
user_service = UserService(pb, scraper)
data: pl.DataFrame = load_data(pb)

node_features = Features(
    tweet=[
        "favorite_count",
        # "retweet_count",
        # "bookmark_count",
        "reply_count",
        "quote_count",
        # "views",
        "source",
        "is_hateful",
    ],
    user=[
        "favourites_count",
        "follower_count",
        "following_count",
        # "number_of_tweets",
        # "listed_count",
        # "is_blue_verified",
        # "friends",
    ],
)


gb = GraphBuilder(data=data, node_features=node_features)
graph: Graph = gb.create_graph(directed=True)

✅ Loading cached dataset - Done.
✅ Loading dataset - Done.
Categorizing source column...

/tmp/ipykernel_622003/1434599103.py:16: DeprecationWarning: preprocess is deprecated. Please call (__call__) the Preprocessor instance after instantiation instead.
  return preprocessor.preprocess()


✅ Categorizing source column - Done.
✅ Preprocessing data - Done.


In [3]:
from collections import defaultdict

freq_table = defaultdict(int)

for node in graph.nodes(data=True):
    freq_table[node[1]['node_label']] += 1

In [4]:
freq_table

defaultdict(int, {1: 35553, 3: 25317, 2: 758, 0: 54})

In [5]:
total_edges = graph.number_of_edges()
total_edges

36365

In [41]:

edge_counts_per_class = defaultdict(int)

for node in graph.nodes(data=True):
    edge_counts_per_class[node[1]['node_label']] += graph.degree[node[0]]
edge_counts_per_class

defaultdict(int, {1: 35553, 3: 36365, 2: 758, 0: 54})

In [44]:
test_table = defaultdict(int)
for edge in graph.edges(data=True):
    node_t = graph.nodes[edge[1]]

    test_table[node_t['node_label']] += 1
test_table

defaultdict(int, {1: 35553, 2: 758, 0: 54})

In [40]:
graph.number_of_nodes()

61682

In [11]:
extremist_possible_edges = freq_table[3] * freq_table[0]
extremist_possible_edges

1367118

In [12]:
neutral_possible_edges = freq_table[3] * freq_table[1]
neutral_possible_edges

900095301

In [13]:
offensive_possible_edges = freq_table[3] * freq_table[2]
offensive_possible_edges

19190286

In [ ]:
assert freq_table[3] + freq_table[0] + freq_table[1] + freq_table[2] == graph.number_of_nodes()


# We check whether the total possible edges from extremist nodes to other nodes matches our calculations
# Since this is a directed bipartite graph, edge calculation is n * m, where n is the number of user nodes
# and m is the number of tweet nodes, m is composed of extremist, neutral, and offensive tweets
assert freq_table[3] * (sum(freq_table[class_id] for class_id in [0, 1, 2])) == extremist_possible_edges + neutral_possible_edges + offensive_possible_edges

In [18]:
total_possible_edges = extremist_possible_edges + neutral_possible_edges + offensive_possible_edges
total_possible_edges

920652705

In [19]:
positive_edges = graph.number_of_edges()
positive_edges

36365

In [ ]:
# We only node degrees of users since edges are directed from users to tweets
assert positive_edges == edge_counts_per_class[3]

In [37]:
negatives_edges = total_possible_edges - positive_edges
negatives_edges

920616340

In [38]:
positive_edges / total_possible_edges

3.949915076825848e-05

In [39]:
negatives_edges / total_possible_edges

0.9999605008492317